In [12]:
import re
from collections import Counter
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from nltk.tokenize import word_tokenize
from torch.utils.data import DataLoader, TensorDataset

In [13]:
df = pd.read_csv(r'D:\stats\Deep_Learning\task\LSTM\sms_spam.csv')
df.head()

,type,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [14]:
tokenized_texts = []
for text in df['text']:
    tokenized_texts.append(word_tokenize(text))

all_tokens = []
for tokens in tokenized_texts:
    all_tokens.extend(tokens)

token_counts = Counter(all_tokens)

vocab = {'<pad>': 0, '<unk>': 1}
for token, _ in token_counts.items():
    if token not in vocab:
        vocab[token] = len(vocab)

In [15]:
encoded_traindf = []
for text in df['text']:
    tokens = word_tokenize(text)
    encoded_traindf.append([vocab.get(token, vocab['<unk>']) for token in tokens])

encoded_traindf[:3]

[[2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  19],
 [25, 26, 19, 27, 28, 29, 30, 19],
 [31,
  32,
  11,
  33,
  34,
  35,
  36,
  37,
  38,
  39,
  40,
  41,
  42,
  43,
  44,
  45,
  46,
  47,
  39,
  37,
  48,
  37,
  49,
  32,
  50,
  51,
  52,
  53,
  54,
  55,
  56,
  57,
  58,
  59,
  60,
  61,
  59]]

In [16]:
MAX_SEQ_LEN = 5

padded_X = []
for sentence in encoded_traindf:
  if len(sentence) > MAX_SEQ_LEN:
    padded_X.append(sentence[:MAX_SEQ_LEN])
  else:
    padded_X.append(sentence + [0]*(MAX_SEQ_LEN-len(sentence)))

padded_X = np.array(padded_X)

In [17]:
df[['label']] = df[['type']].apply(lambda col: pd.Categorical(col).codes)
labels = np.array(df['label'])

In [18]:
indices = np.arange(len(padded_X))
np.random.seed(42)
np.random.shuffle(indices)

split_index = int(len(indices) * 0.8)
train_idx = indices[:split_index]
val_idx = indices[split_index:]

train_set = TensorDataset(torch.from_numpy(padded_X[train_idx]), torch.from_numpy(labels[train_idx]))
val_set = TensorDataset(torch.from_numpy(padded_X[val_idx]), torch.from_numpy(labels[val_idx]))

print('Train size:', len(train_set), '| Validation size:', len(val_set))

Train size: 4459 | Validation size: 1115


In [19]:
batch_size = 4

train_loader = DataLoader(train_set, batch_size, pin_memory=True, shuffle=True)
valid_loader = DataLoader(val_set, batch_size, pin_memory=True, shuffle=False)

In [20]:
class LSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers, patience):
        super(LSTM, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            batch_first=True,
            dropout=0.3,
            num_layers=num_layers
        )

        self.fc = nn.Linear(hidden_dim, 1)
        self.dropout = nn.Dropout(0.3)
        self.sig = nn.Sigmoid()

    def forward(self, x, h):
        x = x.long()
        embeds = self.embedding(x)
        lstm_out, _ = self.lstm(embeds, h)
        lstm_out = lstm_out[:, -1, :]
        lstm_out = self.dropout(lstm_out)
        out = self.fc(lstm_out)
        out = self.sig(out)
        return out

In [21]:
vocab_size = len(vocab)
embedding_dim = 10 
hidden_dim = 3 
num_layers = 2

epochs = 30
lr = 0.001 

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

model = LSTM(vocab_size, embedding_dim, hidden_dim, num_layers, 2)
optimizer = torch.optim.Adam(params = model.parameters(), lr=lr)
model.to(device)

Device: cpu


LSTM(
  (embedding): Embedding(11494, 10)
  (lstm): LSTM(10, 3, num_layers=2, batch_first=True, dropout=0.3)
  (fc): Linear(in_features=3, out_features=1, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (sig): Sigmoid()
)

In [22]:
criterion = nn.BCELoss()

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    train_loss_sum = 0.0
    train_total = 0

    for texts, labels in train_loader:
        texts = texts.to(device)
        labels = labels.to(device)

        bs = labels.size(0)
        h0 = torch.zeros(num_layers, bs, hidden_dim, device=device)
        c0 = torch.zeros(num_layers, bs, hidden_dim, device=device)

        preds = model(texts, (h0, c0)).squeeze()
        loss = criterion(preds, labels.float())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item() * bs
        train_total += bs

    model.eval()
    val_loss_sum = 0.0
    val_total = 0

    with torch.no_grad():
        for texts, labels in valid_loader:
            texts = texts.to(device)
            labels = labels.to(device)

            bs = labels.size(0)
            h0 = torch.zeros(num_layers, bs, hidden_dim, device=device)
            c0 = torch.zeros(num_layers, bs, hidden_dim, device=device)

            preds = model(texts, (h0, c0)).squeeze()
            loss = criterion(preds, labels.float())

            val_loss_sum += loss.item() * bs
            val_total += bs

    epoch_train_loss = train_loss_sum / max(train_total, 1)
    epoch_val_loss = val_loss_sum / max(val_total, 1)

    print(f"Epoch {epoch+1}/{epochs} - Train Loss: {epoch_train_loss:.4f} - Val Loss: {epoch_val_loss:.4f}")

c:\Users\Bhavik\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 1/30 - Train Loss: 0.4463 - Val Loss: 0.3893
Epoch 2/30 - Train Loss: 0.3560 - Val Loss: 0.3065
Epoch 3/30 - Train Loss: 0.2909 - Val Loss: 0.2763
Epoch 4/30 - Train Loss: 0.2568 - Val Loss: 0.2568
Epoch 5/30 - Train Loss: 0.2316 - Val Loss: 0.2431
Epoch 6/30 - Train Loss: 0.2071 - Val Loss: 0.2291
Epoch 7/30 - Train Loss: 0.1924 - Val Loss: 0.2239
Epoch 8/30 - Train Loss: 0.1733 - Val Loss: 0.2220
Epoch 9/30 - Train Loss: 0.1632 - Val Loss: 0.2162
Epoch 10/30 - Train Loss: 0.1507 - Val Loss: 0.2118
Epoch 11/30 - Train Loss: 0.1374 - Val Loss: 0.2094
Epoch 12/30 - Train Loss: 0.1269 - Val Loss: 0.2133
Epoch 13/30 - Train Loss: 0.1243 - Val Loss: 0.2128
Epoch 14/30 - Train Loss: 0.1120 - Val Loss: 0.2109
Epoch 15/30 - Train Loss: 0.1052 - Val Loss: 0.2171
Epoch 16/30 - Train Loss: 0.0962 - Val Loss: 0.2211
Epoch 17/30 - Train Loss: 0.0860 - Val Loss: 0.2243
Epoch 18/30 - Train Loss: 0.0873 - Val Loss: 0.2144
Epoch 19/30 - Train Loss: 0.0777 - Val Loss: 0.2261
Epoch 20/30 - Train L

In [23]:
model.eval()

y_true = []
y_pred = []
y_prob = []

with torch.no_grad():
    for texts, labels in valid_loader: # change to test_loader if you want test metrics
        texts = texts.to(device)
        labels = labels.to(device)
        bs = labels.size(0)
        h0 = torch.zeros(num_layers, bs, hidden_dim, device=device)
        c0 = torch.zeros(num_layers, bs, hidden_dim, device=device)

        probs = model(texts, (h0, c0)).view(-1)          # sigmoid outputs
        preds = (probs >= 0.5).long()                    # threshold

        y_true.extend(labels.view(-1).cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())
        y_prob.extend(probs.cpu().numpy().tolist())



In [24]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score
)
import numpy as np

In [25]:
y_true = np.array(y_true).astype(int)
y_pred = np.array(y_pred).astype(int)
y_prob = np.array(y_prob).astype(float)

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)
rec = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

print("Accuracy :", round(acc, 4))
print("Precision:", round(prec, 4))
print("Recall   :", round(rec, 4))
print("F1-score :", round(f1, 4))
print("Specificity:", round(spec, 4))
print("TP:", tp, "FP:", fp, "TN:", tn, "FN:", fn)


Accuracy : 0.9507
Precision: 0.8676
Recall   : 0.7613
F1-score : 0.811
Specificity: 0.9812
TP: 118 FP: 18 TN: 942 FN: 37
